# Kev: etiquetado de reseñas

Ejemplo mínimo de cómo llamar a Kev, que corre en un pod de RunPod, usando `typesafe_sdk`.

**Antes de correr:**
1. `RUNPOD_API_KEY` y `KEV_API_KEY` en el `.env` de la raíz del repo.
2. Pod levantado con `uv run python deploy_models.py up` (o por la API). Bajarlo con `uv run python deploy_models.py down`.

La URL del pod se le pide a RunPod con `get_active_pod` de `engine/infra.py`, el mismo código que usan la API y `deploy_models.py`.

In [ ]:
import sys
from pathlib import Path

import httpx
import runpod

ROOT = Path("../..").resolve()  # raíz del repo, relativa a este notebook
sys.path.insert(0, str(ROOT))      # para importar engine y settings

from engine.infra import get_active_pod
from settings import get_settings

settings = get_settings()
runpod.api_key = settings.runpod_api_key  # el SDK de RunPod no lee RUNPOD_API_KEY solo

pod = get_active_pod(settings.infra.port)
assert pod is not None, "No hay ningun pod `kev` corriendo: uv run python deploy_models.py up"

BASE_URL = pod["base_url"]
API_KEY = settings.kev_api_key
BASE_URL

In [ ]:
httpx.get(pod["health_url"], timeout=30).status_code  # 502 mientras kev carga, 200 cuando está listo

In [15]:
from typesafe_sdk import Choice, TypeSafeClient

client = TypeSafeClient(
    api_key=API_KEY,
    base_url=BASE_URL,
    model="kev-latest",
    # el SDK comprime con zstd los bodies de más de 1 KB y kev.serve no los sabe leer
    compression=False,
    timeout=120,
)

In [ ]:
SCALE = {
    "1": "Very negative",
    "2": "Negative",
    "3": "Mixed or acceptable",
    "4": "Positive",
    "5": "Very positive",
    "N/A": "The review does not give enough evidence to rate this aspect",
}

questions = {
    "food": Choice(instructions="How does the review rate the food and drinks?", criteria=SCALE),
    "wait_time": Choice(instructions="How does the review rate the speed of service?", criteria=SCALE),
}

review = "The food was delicious, but it took an hour to arrive."
response = client.system_one(state=review, questions=questions)
response.answers